This notebook follows a similar workflow as the Compute FISH distances notebook. Use this notebook once you have optimized your parameters for segmentation and thresholding and batch process all the files at once.

## Import packages

In [1]:
import fishtools as ft
import warnings
import numpy as np
import os
import pandas as pd
from aicsimageio import AICSImage

In [ ]:
# Make a list of all the files you want to process
czi_list=ft.find_czi_files("/Users/agastya.singh/Desktop/4.09.25 IF+FISH WT","Ubtf")

## Batch processing

In [ ]:
num_nuclei_l=[]
num_nuclei_unf_l=[]
spots647_l=[]
spots647_unf_l=[]
spots555_l=[]
spots555_unf_l=[]
file_order=[]
failed_iterations=[]

In [ ]:
for i in range(len(czi_list)):
    path=czi_list[i]

    try:
        # Capturing metadata from the filename. This depends on how you name your files. Not needed.
        c1,c2,c3,cell_stage,condition,file=ft.parse_image_properties(path)
        gene_647=c1
        color_647='yellow'
        gene_555=c2
        color_555="red"
        print("Starting to process:",file)

        # Capturing actual image data from the file. Also correcting for z-depth intensity (z-slices close to the objective are inherently brighter). Getting the scale in microns from the data and splitting the image data into 4 channels. The order of channels will change depending on your aquisition.
        volume_uncorrected,scale=ft.get_data_aicsimageio(path)
        volume=ft.correct_z_intensity(volume_uncorrected)
        ratio = scale[0] / scale[1]
        layer647    = volume[0]
        layer555    = volume[1]
        specklelayer = volume[2]
        dapilayer   = volume[3]
        
        # Segmenting nucleus, calculating spots and segmenting speckles and chromatin.
        nuclei_labels,nuclei_mask,num_nuclei_unf=ft.segment_nucleus(dapilayer,scale_zyx=scale)
        spots_647,num_spots647_unf=ft.identify_fishspots_soph(layer647,spot_name=gene_647,nuclei_mask=nuclei_mask,spot_color=color_647,scale_zyx=scale,intensity_thresh_coeff=1.25) #coeff is x in thresh = x * triangle_thresh. Should be higher for brighter spots and lower for weaker spots.
        spots_555, num_spots555_unf=ft.identify_fishspots_soph(layer555, spot_name=gene_555,nuclei_mask=nuclei_mask,spot_color=color_555,scale_zyx=scale,intensity_thresh_coeff=1.1)
        speckle_labels,speckle_mask=ft.segment_speckles(specklelayer,nuclei_mask,scale)
        chromatin_mask=ft.mask_chromatin(dapilayer,scale)

        # Calculating EDT for distances to lamina and speckles.
        lamina_edt=ft.perform_edt(nuclei_mask,scale)[0]
        speckles_edt=ft.calculate_euclidean_distance_from_speckles(speckle_mask,nuclei_mask,chromatin_mask,scale)

        # Saving data on number of nuclei and FISH spots detected for debugging and sanity checks.
        num_nuclei=len(np.unique(nuclei_labels))-1
        num_spots647=len(spots_647)
        num_spots555=len(spots_555)

        num_nuclei_l.append(num_nuclei)
        num_nuclei_unf_l.append(num_nuclei_unf)
        spots647_l.append(num_spots647)
        spots647_unf_l.append(num_spots647_unf)
        spots555_l.append(num_spots555)
        spots555_unf_l.append(num_spots555_unf)
        file_order.append(file)

        # Overlaying FISH spots on the EDTs calculated to compute the distances. Summarising the results in a nice table.
        summary_df=ft.summary_table(spots_647,nuclei_labels,speckle_labels,lamina_edt,speckles_edt,scale,centroids2=spots_555,centroids1_name=gene_647,centroids2_name=gene_555,path=path)
        
        # Saving the output dataframe.
        save_path_output=os.path.splitext(path)[0]+"_summarydf.tsv"
        summary_df.to_csv(save_path_output,sep="\t",header=True,index=True)
        if len(spots_647)>4 or len(spots_555)>4:
            print("More spots detected in:",file)
        print("End processing",file)
        print(i+1,"/",len(czi_list)," files completed",end='\r', flush=True)

        pass
    except Exception as e:
        print(f"Error at iteration {i} ({file}): {e}")
        failed_iterations.append(file)
        num_nuclei_l.append("error")
        num_nuclei_unf_l.append("error")
        spots647_l.append("error")
        spots647_unf_l.append("error")
        spots555_l.append("error")
        spots555_unf_l.append("error")
        file_order.append(file)
        continue

# Creating a debugging dataframe.
debug_df=pd.DataFrame({'file':file_order,"num_unfiltered_nuclei":num_nuclei_unf_l,"num_filtered_nuclei":num_nuclei_l,"spots647_unfiltered":spots647_unf_l,"spots647_filtered":spots647_l,"spots555_unfiltered":spots555_unf_l,"spots555_filtered":spots555_l})
print(f"\nFailed iterations: {len(failed_iterations)}")
print(failed_iterations)

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(debug_df)

In [5]:
debug_df.to_csv("/Users/agastya.singh/Desktop/4.09.25 IF+FISH WT/debug_df.tsv",sep="\t",header=0)